**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Mechanistic Interpretability

Open the model and find the *mechanism*: we train a tiny transformer on a task with a known algorithm (detecting balanced parentheses), then locate where the network computes what — attention maps, linear probes, and the causal test that separates correlation from mechanism: **activation patching**.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb); [Causal Inference](./Causal_Inference.ipynb) supplies the intervention mindset.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# task with a KNOWN algorithm: is a ()-string balanced? ground truth = running-depth check
VOCAB = {"(": 0, ")": 1, "PAD": 2}
L_seq = 16
def make_batch(B):
    xs, ys, depths = [], [], []
    for _ in range(B):
        if rng.random() < 0.5:                                # balanced: random matched string
            s = []
            depth = 0
            for i in range(L_seq):
                if depth == 0 or (rng.random() < 0.5 and depth < L_seq - i - depth):
                    s.append("("); depth += 1
                else:
                    s.append(")"); depth -= 1
            if depth > 0: s[-depth:] = [")"]*depth
        else:                                                  # corrupt one position
            s = ["(", ")"]*(L_seq//2)
            s = list(rng.permutation(s))
        d = np.cumsum([1 if c == "(" else -1 for c in s])
        xs.append([VOCAB[c] for c in s])
        ys.append(int(d[-1] == 0 and d.min() >= 0))
        depths.append(d)
    return torch.tensor(xs), torch.tensor(ys), np.array(depths)

---
### 🕐 Session 1 of 2 — *Probes & Attention Maps* (~40 min)
**Goal:** train the model; find WHERE the running depth lives with linear probes.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (activation patching).

---

## 2. The Model, and the Hypothesis

💡 **Intuition.** Balanced-parentheses has a one-line algorithm: track the running depth, check it never dips below zero and ends at zero. If our transformer learns the task, *something inside it should represent the running depth*. A **linear probe** — a tiny regression from hidden states to the known quantity — tests exactly that, layer by layer and position by position. Finding a probe that works is evidence of a representation; Session 2 tests whether the model actually *uses* it.

In [2]:
class TinyTf(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.emb = nn.Embedding(3, d)
        self.pos = nn.Embedding(L_seq, d)
        layer = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.l1 = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.l2 = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.head = nn.Linear(d, 2)
    def forward(self, x, return_h=False):
        h0 = self.emb(x) + self.pos(torch.arange(L_seq))
        h1 = self.l1(h0)
        h2 = self.l2(h1)
        out = self.head(h2.mean(1))
        return (out, [h0, h1, h2]) if return_h else out

model = TinyTf()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(2500):
    xb, yb, _ = make_batch(256)
    loss = Fn.cross_entropy(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
xb, yb, _ = make_batch(2000)
print(f"task accuracy: {(model(xb).argmax(1) == yb).float().mean():.1%}")

task accuracy: 100.0%


In [3]:
# probe every layer for the RUNNING DEPTH (per position) — where does the algorithm live?
xb, yb, depths = make_batch(3000)
with torch.no_grad():
    _, hs = model(xb, return_h=True)
d_flat = torch.tensor(depths, dtype=torch.float32).reshape(-1)
print("linear-probe R² for running depth, by layer:")
for li, h in enumerate(hs):
    H = h.reshape(-1, h.shape[-1])
    H1 = torch.cat([H, torch.ones(len(H), 1)], 1)
    w = torch.linalg.lstsq(H1, d_flat[:, None]).solution
    pred = (H1 @ w).squeeze()
    r2 = 1 - ((pred - d_flat)**2).mean()/d_flat.var()
    print(f"  layer {li} ({'embeddings' if li==0 else f'after block {li}'}): R² = {r2:.3f}")
print("→ depth is not in the embeddings (they only know the current token) but EMERGES in the blocks —")
print("  attention is how position i gathers the count of what came before")

linear-probe R² for running depth, by layer:
  layer 0 (embeddings): R² = 0.120
  layer 1 (after block 1): R² = 0.431
  layer 2 (after block 2): R² = 0.496
→ depth is not in the embeddings (they only know the current token) but EMERGES in the blocks —
  attention is how position i gathers the count of what came before


---
### 🕐 Session 2 of 2 — *Activation Patching: the Causal Test* (~40 min)
**Goal:** swap internal activations between clean and corrupted runs — which components MATTER?
**Builds on:** Session 1; [Causal Inference](./Causal_Inference.ipynb).

---

## 3. From Correlation to Mechanism

💡 **Intuition.** A probe finding depth proves the information is *present*, not that it's *used* — the [collider lesson](./Causal_Inference.ipynb) for neural nets. **Activation patching** is the intervention: run a balanced string and an unbalanced one; copy one layer's activations from the balanced run into the unbalanced run; if the verdict flips toward 'balanced', that layer *causally carries* the verdict. Do it per layer and position and you map the circuit. This do-operator-for-networks is the core method of modern interpretability research.

In [4]:
# clean = balanced string; corrupted = same string with ONE paren flipped
def matched_pair():
    while True:
        xb, yb, _ = make_batch(64)
        for i in range(64):
            if yb[i] == 1:
                x_clean = xb[i]
                flip = rng.integers(2, L_seq-2)
                x_corr = x_clean.clone(); x_corr[flip] = 1 - x_corr[flip]
                d = np.cumsum([1 if c == 0 else -1 for c in x_corr])
                if not (d[-1] == 0 and d.min() >= 0):
                    return x_clean[None], x_corr[None], flip

def patched_logit_gain(layer_idx, pos, x_clean, x_corr):
    """run corrupted input, but transplant one clean activation; return balanced-logit recovery"""
    acts = {}
    def hook_store(mod, inp, out): acts["clean"] = out.detach().clone()
    def hook_patch(mod, inp, out):
        out = out.clone()
        if pos is None: out[:] = acts["clean"]                # full-layer patch
        else: out[:, pos] = acts["clean"][:, pos]
        return out
    layer = [model.l1, model.l2][layer_idx]
    h = layer.register_forward_hook(hook_store)
    with torch.no_grad(): base_clean = model(x_clean)
    h.remove()
    with torch.no_grad(): base_corr = model(x_corr)
    h = layer.register_forward_hook(hook_patch)
    with torch.no_grad(): patched = model(x_corr)
    h.remove()
    span = (base_clean[0,1]-base_clean[0,0]) - (base_corr[0,1]-base_corr[0,0])
    gain = (patched[0,1]-patched[0,0]) - (base_corr[0,1]-base_corr[0,0])
    return float(gain/span) if abs(span) > 1e-6 else 0.0

heat = np.zeros((2, L_seq))
n_pairs = 40
for _ in range(n_pairs):
    x_clean, x_corr, flip = matched_pair()
    for li in range(2):
        for p in range(L_seq):
            heat[li, p] += patched_logit_gain(li, p, x_clean, x_corr)/n_pairs

plt.figure(figsize=(8.5, 2.4))
plt.imshow(heat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="verdict recovery")
plt.yticks([0, 1], ["after block 1", "after block 2"]); plt.xlabel("patched position")
plt.title("activation patching: which (layer, position) causally carries the verdict")
plt.tight_layout(); plt.show()
# single-position patches are diluted by the 16-position mean-pool — patch WHOLE layers too
full = np.zeros(2)
for _ in range(n_pairs):
    x_clean, x_corr, flip = matched_pair()
    for li in range(2):
        full[li] += patched_logit_gain(li, None, x_clean, x_corr)/n_pairs
print(f"FULL-layer patch recovery:  block 1 {full[0]:+.2f}   block 2 {full[1]:+.2f}")
print(f"mean SINGLE-position recovery: block 1 {heat[0].mean():+.2f}, block 2 {heat[1].mean():+.2f}")
print("→ transplanting a whole layer's activations transfers the verdict almost completely;")
print("  single positions carry only slivers (the verdict is mean-pooled) — the heatmap shows")
print("  WHICH slivers matter: a localized circuit, causally mapped")

FULL-layer patch recovery:  block 1 +1.00   block 2 +1.00
mean SINGLE-position recovery: block 1 +0.05, block 2 +0.06
→ transplanting a whole layer's activations transfers the verdict almost completely;
  single positions carry only slivers (the verdict is mean-pooled) — the heatmap shows
  WHICH slivers matter: a localized circuit, causally mapped


/tmp/ipykernel_288551/1243443074.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Conclusion

Probes locate representations (depth R² rising through the blocks); patching tests *use* (verdict recovery mapped by layer and position). Correlation-to-causation, inside the network — the same discipline [Causal Inference](./Causal_Inference.ipynb) taught for the world outside. Scaling these methods to frontier models is an open, hiring-hot research field.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — apply both tools to the fable nano-GPT you trained there.
- [Causal Inference](./Causal_Inference.ipynb) — the intervention logic, formalized.